# Knife-edge profile: derivative of `weird_two_peak_mode.txt`

This uses the same method as `../fundamental/derivada_perfil.ipynb` and `../modo_semaforo_detapado_a_destapado/`.
The beam profile along the knife's travel is $I(t) = dP/dt$. This mode shows two lobes.

**Caveat:** the knife was moved by hand, so its speed is neither known nor constant. The x axis is time,
and the positions and widths are only qualitative.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

# PM100D log: 2 header lines, then "dd/mm/yyyy hh:mm:ss,fff \t value \t W" (decimal comma)
t, P = [], []
with open('weird_two_peak_mode.txt', encoding='latin-1') as fh:
    for line in fh.read().splitlines()[2:]:
        parts = line.split('\t')
        if len(parts) < 2:
            continue
        t.append(datetime.strptime(parts[0].strip(), '%d/%m/%Y %H:%M:%S,%f').timestamp())
        P.append(float(parts[1].replace(',', '.')))
t = np.array(t) - t[0]          # s
P = np.array(P) * 1e3           # mW
t_raw, P_raw = t.copy(), P.copy()
print(f'{len(t)} points')

## The artifact: the power meter switching range

This is the same problem as in the semáforo file. The meter is on `Range Auto`, and at about 4.7 mW it switches range:
one reading jumps (+0.94 mW here, because the power was rising fast), then nothing is logged for about 0.4 s.

The mismatch across the gap is **not a power offset, it's lost time**. If you extrapolate a straight line from
each side of the gap, the mismatch divided by the local slope comes out to about 0.2 s, both here and in the
semáforo file, even though the slopes differ by 4×. So the meter's clock loses about 0.2 s during the switch.

The fix:
1. drop the jumped reading,
2. shift the timestamps after the switch forward by that amount.

To avoid all this next time, **fix the range manually** before scanning.

In [ ]:
gap = np.argmax(np.diff(t))                  # index right before the dead time
t_switch = t[gap]
print(f'dead time of {t[gap+1] - t[gap]:.3f} s at t = {t_switch:.2f} s')

t, P = np.delete(t, gap), np.delete(P, gap)  # the jumped reading, logged just before the dead time

before = (t > t_switch - 0.8) & (t < t_switch)
after = (t > t_switch) & (t < t_switch + 1.2)
pb = np.polyfit(t[before], P[before], 1)
pa = np.polyfit(t[after], P[after], 1)
tc = t_switch + 0.2
lost = (np.polyval(pa, tc) - np.polyval(pb, tc)) / ((pa[0] + pb[0]) / 2)
t[t > t_switch] += lost
print(f'time lost in the range switch: {lost:.3f} s (added back)')

# before ~3 s the knife isn't moving yet (the power just drifts down a little),
# after ~28 s the beam is fully uncovered
t_min, t_max = 3, 28
keep = (t > t_min) & (t < t_max)
t, P = t[keep], P[keep]

## Savitzky–Golay derivative

In [ ]:
dt = 0.02

def sg_deriv(tt, PP, win_s, order=3):
    tu = np.arange(tt[0], tt[-1], dt)
    n = int(round(win_s / dt)) | 1   # window length has to be odd
    return tu, savgol_filter(np.interp(tu, tt, PP), n, order, deriv=1, delta=dt)

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].plot(t_raw, P_raw, '.', color='0.7', ms=2, label='raw')
ax[0].plot(t, P, 'k.', ms=2, label='corrected + cut')
ax[0].axvline(t_switch, color='r', ls=':', lw=1)
ax[0].set_ylabel('P [mW]')
ax[0].legend()
tu_raw, d_raw = sg_deriv(t_raw, P_raw, 1.0)
ax[1].plot(tu_raw, d_raw, color='0.7', lw=1, label='raw, SG 1 s')
for win in [0.6, 1.0, 1.5]:
    ax[1].plot(*sg_deriv(t, P, win), lw=1, label=f'corrected, SG {win} s')
ax[1].axvline(t_switch, color='r', ls=':', lw=1)
ax[1].set_xlabel('t [s]')
ax[1].set_ylabel('dP/dt [mW/s]')
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

tu, dPdt = sg_deriv(t, P, 1.0)
print(f'area under dP/dt = {np.trapezoid(dPdt, tu):.3f} mW   vs   step = {P[-30:].mean() - P[:30].mean():.3f} mW')

## Two-lobe fit

The second lobe has the same long tail as the fundamental: a sharp rise followed by a slow decay. That shape is
most likely the hand slowing down near the end of the scan. So we compare two fits:
- two plain Gaussians,
- a Gaussian plus a split Gaussian (different widths on each side) for the second lobe.

Neither fit reproduces the flat stretch between the lobes (about 6–9 s): the real profile doesn't dip there as much
as two Gaussians do. A model with four widths fits better, but it's degenerate: the first lobe grows a 9 s tail
that fills that stretch, so its split of the power between the lobes means nothing. The honest takeaway is
qualitative: two lobes about 5 s apart, and the second carries most of the power
(3–6× more than the first, depending on the model).

In [ ]:
def gauss(t, a, t0, w):
    return a * np.exp(-2 * (t - t0)**2 / w**2)

def split_gauss(t, a, t0, wl, wr):
    return a * np.exp(-2 * (t - t0)**2 / np.where(t < t0, wl, wr)**2)

# no constant baseline: dP/dt is ~0 before the knife moves, and a free constant
# just soaks up the slow tail (it stole ~3 mW of the lobes' power)
def two_gauss(t, a1, t1, w1, a2, t2, w2):
    return gauss(t, a1, t1, w1) + gauss(t, a2, t2, w2)

def gauss_split(t, a1, t1, w1, a2, t2, wl, wr):
    return gauss(t, a1, t1, w1) + split_gauss(t, a2, t2, wl, wr)

p2, c2 = curve_fit(two_gauss, tu, dPdt, p0=[1.5, 6, 2.5, 2, 11, 6])
ps, cs = curve_fit(gauss_split, tu, dPdt, p0=[1.5, 6, 2.5, 2, 11, 4, 7])

def power(a, w):   # area of a*exp(-2(t-t0)^2/w^2)
    return a * w * np.sqrt(np.pi / 2)

e2, es = np.sqrt(np.diag(c2)), np.sqrt(np.diag(cs))
print(f'two gaussians          (rms residual {np.std(dPdt - two_gauss(tu, *p2)):.3f} mW/s)')
print(f'  lobe 1: t0 = {p2[1]:.2f} ± {e2[1]:.2f} s,  w = {p2[2]:.2f} ± {e2[2]:.2f} s,  power = {power(p2[0], p2[2]):.2f} mW')
print(f'  lobe 2: t0 = {p2[4]:.2f} ± {e2[4]:.2f} s,  w = {p2[5]:.2f} ± {e2[5]:.2f} s,  power = {power(p2[3], p2[5]):.2f} mW')
print(f'gaussian + split       (rms residual {np.std(dPdt - gauss_split(tu, *ps)):.3f} mW/s)')
print(f'  lobe 1: t0 = {ps[1]:.2f} ± {es[1]:.2f} s,  w = {ps[2]:.2f} ± {es[2]:.2f} s,  power = {power(ps[0], ps[2]):.2f} mW')
print(f'  lobe 2: t0 = {ps[4]:.2f} ± {es[4]:.2f} s,  w = {ps[5]:.2f} / {ps[6]:.2f} s,  '
      f'power = {power(ps[3], (ps[5] + ps[6]) / 2):.2f} mW')

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for axis, f, p, title in [(ax[0], two_gauss, p2, 'two gaussians'), (ax[1], gauss_split, ps, 'gaussian + split gaussian')]:
    axis.plot(tu, dPdt, 'k-', lw=1, label='dP/dt (SG 1 s)')
    axis.plot(tu, f(tu, *p), 'r-', label=title)
    axis.plot(tu, gauss(tu, *p[0:3]), '--', lw=1)
    axis.plot(tu, f(tu, *p) - gauss(tu, *p[0:3]), '--', lw=1)
    axis.set_ylabel('dP/dt [mW/s]')
    axis.legend()
ax[1].set_xlabel('t [s]')
plt.tight_layout()
plt.show()